# 1. Introduction
*   **Weights & Biases Link:** TODO [Insert the link to your W&B report/view here] [1].
*   **Tools & Sources Used:**
    - NotebookLM: Summarizing course materials and project description, Creating Template of Python Notebook
    - Gemini: Coding support / Learning help
    - Claude: Coding support / Coding Coach


# 2. Setup
*State and justify your setup decisions here [1].*

In [5]:
!pip install datasets gensim wandb nltk fasttext

  Using cached fasttext-0.9.3-cp311-cp311-macosx_14_0_universal2.whl
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [fasttext]


In [1]:
# Import all necessary libraries (e.g., PyTorch, Lightning, Hugging Face Datasets, Weights & Biases) [2, 4, 5]
import torch
from datasets import load_dataset
import wandb
from gensim.models import FastText # https://radimrehurek.com/gensim/models/fasttext.html # https://medium.com/@abhishekjainindore24/fasttext-embedding-technique-9b5d1fba4b50 
import gensim.downloader as api
from nltk.tokenize import word_tokenize
import nltk
import numpy as np
import random
import fasttext

In [2]:
# Reproducibility with seed 42
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [3]:
# Login to W&B
wandb.login()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /Users/janis.kneubuehler/.netrc.
wandb: Currently logged in as: janis-kneubuehler (janis-kneubuehler-hochschule-luzern) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

# 3. Preprocessing & Data Loading
**Decisions:**
- Tokenization: I use nltk.word_tokenize as recommended in the project discussion and demonstrated in the exercise of SW02.
- Lowercase: I convert to lowercase to reduce the vocabulary size, therefore we have less unknown words
- Removal of punctuation: I remove punctuation for reducing vocabulary size and because they have no semantic meaning. with the nltk.word_tokenize they are separate tokens and easy to remove. 
- Removal of stopwords: I don't remove stopwerds, because default stopwords like "in", "on", etc. are critical for understanding physical relationships. With removal of them the project wouldn't make any sense.
- Stemming and Lemmatization: I skip stemming and lemmatizing because the word2vec is huge and has different word forms in it. Stemming would maybe perform worse, because some "not-real words" wouldn't be recognized by word2vec.
- Cleaning unknown words: I map unkknown words into an <unk> token. This was suggested by Gemini as it is a simple way to prevent KeyErros while training.  
- Format cleaning: No html removal necessary, because it is already cleaned in the dataset. (see Data Exploration)
- Truncation: No truncation made due to TODO
- Feature selection: TODO
- Input format: I concatenate the goal with both solution and a separator, so from one row i get two: goal + <SEP> + solution1 and goal + <SEP> + solution2. TODO
- Label format: A boolean if solution1 or solution2 is correct.
- Batching, padding: TODO
- Vocabulary, Embedding: TODO

In [19]:
# Load the PIQA dataset from revision because dataset scripts are no longer supported
# Use splits like specified in Course Project Slides
train = load_dataset('ybisk/piqa', split='train[:-1000]', revision='refs/convert/parquet')
valid = load_dataset('ybisk/piqa', split='train[-1000:]', revision='refs/convert/parquet')
test = load_dataset('ybisk/piqa', split='validation', revision='refs/convert/parquet')

In [20]:
import re

# Exploring the data
print(f"training: {len(train)}, validation: {len(valid)}, test: {len(test)}")

print("Exploring some random rows in dataset:")
print(train[0])
print(train[12])
print(train[123])
print(train[4563])
print(train[13245]) 

# Count rows with html (Regex was generated with Gemini and validated by me with https://regex101.com/)
goals_with_html = train.filter(lambda x: re.search(r'</?[^>]+>', x['goal'], re.IGNORECASE))
sols1_with_html = train.filter(lambda x: re.search(r'</?[^>]+>', x['sol1'], re.IGNORECASE))
sols2_with_html = train.filter(lambda x: re.search(r'</?[^>]+>', x['sol2'], re.IGNORECASE))

print(f"Number of rows with html (for format cleaning): {len(goals_with_html)}, {len(sols1_with_html)}, {len(sols2_with_html)}")

training: 15113, validation: 1000, test: 1838
Exploring some random rows in dataset:
{'goal': "When boiling butter, when it's ready, you can", 'sol1': 'Pour it onto a plate', 'sol2': 'Pour it into a jar', 'label': 1}
{'goal': 'how do you open a capri-sun', 'sol1': 'open the straw attached to the juice, and then stick it in the small hole at the front of the pouch.', 'sol2': 'cut the top off with scissors', 'label': 0}
{'goal': 'How do you open a gift without ripping the paper?', 'sol1': 'Hold gift over boiling water to let the steam release the tape adhesive.', 'sol2': 'Hold gift over cold water to let the steam release the tape adhesive.', 'label': 0}
{'goal': 'how to make fancy piped flowers', 'sol1': 'use molded ganache', 'sol2': 'use Russian piping tips', 'label': 1}
{'goal': 'How do you keep steel wool and cotton dry?', 'sol1': 'Place the make-shift fire starter inside a film canister.', 'sol2': 'Place the make-shift fire starter inside your pocket.', 'label': 0}
Number of rows wi

In [21]:
# Workaround on mac for SSL certificate error (Code by Gemini)
import os
import ssl
import certifi

# 1. Point the notebook to your trusted certificates
os.environ['SSL_CERT_FILE'] = certifi.where()
os.environ['REQUESTS_CA_BUNDLE'] = certifi.where()

# 2. Force Python to allow unverified contexts globally (The safety net)
try:
    _create_unverified_https_context = ssl._create_unverified_context
except AttributeError:
    pass
else:
    ssl._create_default_https_context = _create_unverified_https_context

print("SSL Bypass active!")

SSL Bypass active!


In [22]:
# Load word2vec embeddings
wv = api.load('word2vec-google-news-300')
print(f'Vector size: {wv.vector_size}, Vocabulary size: {len(wv.index_to_key)}')

# code from here: https://radimrehurek.com/gensim/auto_examples/tutorials/run_word2vec.html 
for index, word in enumerate(wv.index_to_key):
    if index == 10:
        break
    print(f"word #{index}/{len(wv.index_to_key)} is {word}")


Vector size: 300, Vocabulary size: 3000000
word #0/3000000 is </s>
word #1/3000000 is in
word #2/3000000 is for
word #3/3000000 is that
word #4/3000000 is is
word #5/3000000 is on
word #6/3000000 is ##
word #7/3000000 is The
word #8/3000000 is with
word #9/3000000 is said


In [23]:
import string

def preprocess(text):
    # lowercase all words
    text = text.lower()
    # tokenize
    tokens = nltk.word_tokenize(text)
    # remove punctuations
    tokens = [token for token in tokens if token not in string.punctuation]

    return tokens

def preprocess_data(data):
    data['goal_tokens'] = preprocess(data['goal'])
    data['sol1_tokens'] = preprocess(data['sol1'])
    data['sol2_tokens'] = preprocess(data['sol2'])
    
    # concat goal with input
    data['input1_tokens'] = data['goal_tokens'] + ['<sep>'] + data['sol1_tokens']
    data['input2_tokens'] = data['goal_tokens'] + ['<sep>'] + data['sol2_tokens']
    
    return data

train = train.map(preprocess_data)
valid = valid.map(preprocess_data)
test = test.map(preprocess_data)

# TODO delete all that is not necessary like goal_tokens alone
print(train[0])


Map:   0%|          | 0/15113 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1838 [00:00<?, ? examples/s]

{'goal': "When boiling butter, when it's ready, you can", 'sol1': 'Pour it onto a plate', 'sol2': 'Pour it into a jar', 'label': 1, 'goal_tokens': ['when', 'boiling', 'butter', 'when', 'it', "'s", 'ready', 'you', 'can'], 'sol1_tokens': ['pour', 'it', 'onto', 'a', 'plate'], 'sol2_tokens': ['pour', 'it', 'into', 'a', 'jar'], 'input1_tokens': ['when', 'boiling', 'butter', 'when', 'it', "'s", 'ready', 'you', 'can', '<sep>', 'pour', 'it', 'onto', 'a', 'plate'], 'input2_tokens': ['when', 'boiling', 'butter', 'when', 'it', "'s", 'ready', 'you', 'can', '<sep>', 'pour', 'it', 'into', 'a', 'jar']}


**Decisions specific for Vocabulary:**
- I only use training data for the vocabulary to prevent data leakage
- Not used torchtext because it was not part of our course material

TODO maybe delete this block here

In [24]:
class Vocabulary:
    def __init__(self):
        self.word2idx = {'<pad>': 0, '<unk>': 1, '<sep>': 2}
        self.idx2word = {0: '<pad>', 1: '<unk>', 2: '<sep>'}
        self.idx = 3

    def build_vocab(self, dataset):
        for example in dataset:
            for token in example['goal_tokens'] + example['sol1_tokens'] + example['sol2_tokens']: # Add all tokens to vocabulary
                if token not in self.word2idx:
                    self.word2idx[token] = self.idx
                    self.idx2word[self.idx] = token
                    self.idx += 1

    def encode(self, tokens):
        encoded = []
        unk_count = 0
        for token in tokens:
            if token in self.word2idx:
                encoded.append(self.word2idx[token])
            else:
                encoded.append(self.word2idx['<unk>'])
                unk_count += 1
        return encoded, unk_count

vocab = Vocabulary()
vocab.build_vocab(train)
print(f"Total unique words in Training Vocabulary: {vocab.idx}")

def encode_data(example):
    enc1, unk1 = vocab.encode(example['input1_tokens'])
    enc2, unk2 = vocab.encode(example['input2_tokens'])
    
    example['input1_ids'] = enc1
    example['input2_ids'] = enc2
    
    example['unk_count'] = unk1 + unk2
    return example

# 3. Apply the encoding to all splits
train = train.map(encode_data)
valid = valid.map(encode_data)
test = test.map(encode_data)

# 4. The Required Analysis: How many UNKs did we get?
train_unks = sum(train['unk_count'])
valid_unks = sum(valid['unk_count'])
test_unks = sum(test['unk_count'])

print(f"<unk> tokens in Train: {train_unks} (must be 0)")
print(f"<unk> tokens in Valid: {valid_unks}")
# I don't analyse unk on Test set to not get biased

Total unique words in Training Vocabulary: 15766


Map:   0%|          | 0/15113 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1838 [00:00<?, ? examples/s]

<unk> tokens in Train: 0 (must be 0)
<unk> tokens in Valid: 781


**Decisions specific for collate:**
- I only use training data for the vocabulary to prevent data leakage
- Padding: TODO why pad_sequence
- Batching: TODO not sure yet

TODO maybe delete this block here

In [ ]:
import torch
from torch.utils.data import DataLoader
from torch.nn.utils.rnn import pad_sequence

# TODO not finished yet

def collate_fn(batch):
    input1_list = []
    input2_list = []
    labels = []
    
    for example in batch:
        input1_list.append(torch.tensor(example['input1_ids']))
        input2_list.append(torch.tensor(example['input2_ids']))
        
        labels.append(example['label'])
        
    input1_padded = pad_sequence(input1_list, batch_first=True, padding_value=0)
    input2_padded = pad_sequence(input2_list, batch_first=True, padding_value=0)
    
    labels_tensor = torch.tensor(labels, dtype=torch.long)
    
    return {
        'input1': input1_padded, 
        'input2': input2_padded, 
        'labels': labels_tensor
    }

# 32 is a standard starting batch size, but you can adjust based on your GPU
# TODO fix this
batch_size = 32 

# We shuffle the train loader so the model doesn't memorize the order, 
# but we do NOT shuffle valid and test loaders.
train_loader = DataLoader(train, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
valid_loader = DataLoader(valid, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)
test_loader  = DataLoader(test, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

first_batch = next(iter(train_loader))
print("Input 1 Batch Shape:", first_batch['input1'].shape) # (batch_size, sequence_length)
print("Labels Batch Shape:", first_batch['labels'].shape)  # (batch_size)

Input 1 Batch Shape: torch.Size([32, 77])
Labels Batch Shape: torch.Size([32])


# 4. Model Definition
*State and justify your architectural decisions here [1].*

**Architectures to Implement [7, 8]:**
1.  **Architecture 1:** Pre-trained Word Embeddings (choose word2vec, GloVe, or fastText) + a 2-layer classifier with a ReLU non-linearity. Train *only* the classifier.
2.  **Architecture 2:** A 2-layer RNN (use PyTorch's LSTM or GRU) + a 2-layer classifier with a ReLU. Use the exact same word embeddings from Architecture 1 for the input layer, but train the whole network end-to-end.

In [10]:
# Define Architecture 1 (Embeddings + Classifier)
class EmbeddingClassifier(torch.nn.Module):
    # Your implementation
    pass

# Define Architecture 2 (RNN + Classifier)
class RNNClassifier(torch.nn.Module):
    # Your implementation
    pass


# 5. Training
*State and justify your training decisions here [1].*

*   Discuss your chosen loss function and optimization algorithm.
*   Detail the hyperparameters you are using.


In [11]:
# Initialize Weights & Biases tracking
# wandb.init(project="piqa-project")

# Training loop for Architecture 1
# Training loop for Architecture 2

# Ensure you are actively tracking experiments, parameters, and code versions so your results are reproducible [9].

# 6. Evaluation

In [12]:
# Add your code to evaluate both architectures on the validation set
# Add your code for the final evaluation on the test set

# 7. Conclusion & Interpretation
*State and justify your conclusions here [1].*

*   Interpret your results. Discuss the specialties of each model type, how they handled the physical commonsense reasoning task, and summarize your findings [1, 9].